# Sparkle A/B — legacy ECCO recipe vs square-before-interp

The grad(x)² and squared kinematic fields showed pixel 'sparkle':
isolated WHITE (anomalously small) values on log-scaled maps.  Root
cause (confirmed here): the ECCO gradient recipe computes finite
differences on the staggered points, 2-point-interpolates them to
the cell centre, rotates, THEN squares.  The 2-point mean has a
null space at the grid scale — at a local extremum the two flanking
differences are equal-and-opposite and cancel — so the centred
|grad|² is orders of magnitude below its one-sided gradients.  Not
a precision artifact (store ≡ live to ~1e-7).

The fix (now the production implementation,
`native_gradient.calculate_grad_squared_tracer` and
`native_gradient.kinematic_invariants`): square each difference ON
its native C-grid point, then move only the non-negative squares —
a mean of non-negatives cannot cancel.  |grad|², σ², ζ², W are
rotation-invariant, so no CS/SN is needed.  See
[MITgcm horizontal grid](
https://mitgcm.readthedocs.io/en/latest/algorithm/horiz-grid.html)
and docs/Fields.md.

Figure layout — one figure per squared field:

| | col 1 (raw) | (∂/∂x)² | (∂/∂y)² | final |
|---|---|---|---|---|
| rows 1-2 | ECCO recipe, full + 200×200 km zoom | | | |
| rows 3-4 | square-first, full + zoom | | | |

**Component-column caveat**: ECCO components are geographic
(zonal/meridional); square-first components are MODEL-axis (the
geographic split needs the cross term the squares destroy).  On
rotated LLC faces (e.g. the Gulf Stream) the two appear SWAPPED —
verified corr(ECCO dx², sq-first dy²) ≈ 0.9 vs 0.5 same-name.
The FINAL column is rotation-invariant and directly comparable.
Sequential colormaps only; one shared norm per column.

## Section 1 — Grid

Only the stitched-grid coordinates are needed (the A/B is entirely
live; store-vs-live consistency lives in the validation notebooks).

In [ ]:
# Section 1: shared 2D grid (XC/YC) for stitching + slicing.
import numpy as np
import matplotlib.pyplot as plt
import cmocean.cm as cmo

import dbof.io.filesystems as filesystems
import dbof.global_dataset_creation.zarr_grid_global as zarr_grid

S3_ENDPOINT = "https://s3-west.nrp-nautilus.io"
PIPELINE = "SURF"
DATE = "2012-11-09 12:00:00"   # single validation timestep

fs_grid, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
grid_reader = zarr_grid.GlobalGridZarrReader(
    bucket="dbof", folder="LLC4320_GRID_2D",
    dataset_name="llc4320_grid.zarr", fs=fs_grid,
)
XC, YC = grid_reader.lon, grid_reader.lat
print(f"grid: XC {XC.shape}")

## Section 2 — Region selection

Any region with a `zoom` anchor works; re-run from here after
changing it.

In [ ]:
# Section 2: pick the region.
from dbof.plotting import regions

REGION       = "gulf_stream"   # <-- change me, re-run from here
ZOOM_HALF_KM = 100.0           # 200x200 km zoom box

_zoomable = [n for n, r in regions.REGIONS.items() if "zoom" in r]
assert REGION in _zoomable, f"pick one of {_zoomable}"
print(f"region: {REGION}  (options: {_zoomable})")

## Section 3 — Live fields: both variants, lazily

- **ECCO variant** (legacy recipe) is reconstructed INLINE here —
  it no longer exists in the codebase (grad_*2, strain_mag and
  okubo_weiss are square-first in production since the sparkle
  fix).
- **sq-first variant** = the production functions themselves.

In [ ]:
# Section 3a: snapshot + lazy fields for both variants.
import dbof.preprocessing.calculate_fields as calculate_fields
import dbof.utils.native_gradient as ng
from dbof.cli.generate_global import load_snapshot
from dbof.global_dataset_creation.data_sources import get_data_source
from dbof.global_dataset_creation.grid_setup import set_up_grid
from dbof.utils.faces_to_latlon import stitch_and_mask

ds_grid, land_mask, xgrid = set_up_grid(PIPELINE, None)
ds_raw, ds_merge, it = load_snapshot(
    PIPELINE, DATE, ds_grid, ["Theta", "Salt", "Eta", "U", "V"],
    surface_only=False, data_source=get_data_source(PIPELINE),
)
print(f"OSN iteration {it}")

rho = calculate_fields.potential_density(ds_merge)
b = calculate_fields.buoyancy_of_field(ds_merge)

# Tracer sources per final field: (raw stage name, source array).
TRACERS = {
    "gradtheta2": ("Theta", ds_merge.Theta),
    "gradsalt2": ("Salt", ds_merge.Salt),
    "gradeta2": ("Eta", ds_merge.Eta),
    "gradrho2": ("rho_theta", rho),
    "gradb2": ("buoyancy", b),
}
# Production sq-first finals (the canonical functions).
SQF_FN = {
    "gradtheta2": calculate_fields.grad_theta2,
    "gradsalt2": calculate_fields.grad_salt2,
    "gradeta2": calculate_fields.grad_eta2,
    "gradrho2": calculate_fields.grad_rho2,
    "gradb2": calculate_fields.grad_b2,
}


def _sqfirst_parts(da):
    """Model-axis squared components, square-BEFORE-interp.

    Mirrors the stencils of
    ``ng.calculate_grad_squared_tracer`` (their sum IS that
    function); exposed separately only for the component columns.
    Inputs: da (DataArray at tracer points).
    Outputs: (dx2_c, dy2_c) at tracer points, lazy.
    Generated by LH and Claude
    """
    dx = xgrid.diff(da, 'X') / ds_merge.dxC
    dy = xgrid.diff(da, 'Y') / ds_merge.dyC
    return (xgrid.interp(dx ** 2, 'X', boundary='fill'),
            xgrid.interp(dy ** 2, 'Y', boundary='fill'))


live_map = {}
for _f, (_raw, _da) in TRACERS.items():
    live_map[_raw] = _da
    # ECCO variant (legacy recipe, inline): geographic components,
    # squared AFTER the centre interpolation + rotation.
    _gx, _gy = ng.calculate_native_gradient_tracer(
        _da, ds_merge, grid=xgrid)
    live_map[f"{_f}_dx2_ecco"] = _gx ** 2
    live_map[f"{_f}_dy2_ecco"] = _gy ** 2
    live_map[f"{_f}_ecco"] = _gx ** 2 + _gy ** 2
    # sq-first variant: production function + model-axis parts.
    _px, _py = _sqfirst_parts(_da)
    live_map[f"{_f}_dx2_sqf"] = _px
    live_map[f"{_f}_dy2_sqf"] = _py
    live_map[f"{_f}_sqfirst"] = SQF_FN[_f](ds_merge, xgrid)

# Kinematics.  ECCO variant inline from the centred Jacobian;
# sq-first = the production strain/okubo_weiss (C-grid-native
# invariants, shared).
jac = calculate_fields.compute_velocity_jacobian(ds_merge, xgrid)
_sn = jac.du_dx - jac.dv_dy
_ss = jac.du_dy + jac.dv_dx
_zeta = jac.dv_dx - jac.du_dy
live_map["strain_mag_ecco"] = np.sqrt(_sn ** 2 + _ss ** 2)
live_map["okubo_weiss_ecco"] = _sn ** 2 + _ss ** 2 - _zeta ** 2

inv = ng.kinematic_invariants(ds_merge.U, ds_merge.V, ds_merge,
                              xgrid)
_mag, _, _ = calculate_fields.strain(ds_merge, xgrid, jacobian=jac,
                                     invariants=inv)
live_map["strain_mag_sqfirst"] = _mag
live_map["okubo_weiss_sqfirst"] = (
    calculate_fields.okubo_weiss_parameter(ds_merge, xgrid,
                                           invariants=inv))
print(f"{len(live_map)} lazy live fields")

In [ ]:
# Section 3b: batch-stitch and slice to REGION only.
BATCH = 4
mask = {"_land_mask": (ds_merge.hFacC == 0)}
names = list(live_map)
live_region = {}
for i0 in range(0, len(names), BATCH):
    grp = names[i0:i0 + BATCH]
    ds_conv = ds_raw.assign({n: live_map[n] for n in grp})[grp]
    chw = stitch_and_mask(ds_conv, grp, mask)
    for k, n in enumerate(grp):
        live_region[n] = regions.select_region(chw[k], XC, YC,
                                               REGION)
    del chw
    print(f"stitched + sliced: {grp}")
print(f"{len(live_region)} fields ready")

## Section 4 — Plotting helper (inlined)

Notebook-local by design (LH decision) — this layout exists only
for this A/B.  Sequential colormaps only (diverging maps hide
speckle in their white midpoint); one shared norm per column; log
norms for non-negative columns with the TOP end unclipped so
extreme pixels stay visible; exact zeros clipped to the bottom
colour (not gray, which would read as land).

In [ ]:
# Section 4: 4-row (ECCO / sq-first x full / zoom) stage grid.
import matplotlib.colors as mcolors
from dbof.plotting.pipeline_grids import LAND_COLOR

ROW_LABELS = ("ECCO (full)", "ECCO (zoom)",
              "sq-first (full)", "sq-first (zoom)")
ROBUST_PCT = (1.0, 99.9)
PANEL_W, PANEL_H = 4.4, 3.1


def _stage_norm(arrays):
    """Shared colour norm for one column: log when non-negative
    (zeros allowed — clipped at draw time), robust linear otherwise.
    Inputs: arrays (list of np.ndarray or None).
    Outputs: matplotlib Normalize/LogNorm.
    Generated by LH and Claude
    """
    vals = np.concatenate(
        [a[np.isfinite(a)].ravel() for a in arrays
         if a is not None])
    if vals.size == 0:
        return mcolors.Normalize(0.0, 1.0)
    pos = vals[vals > 0]
    if vals.min() >= 0 and pos.size:
        lo = np.percentile(pos, ROBUST_PCT[0])
        hi = pos.max()            # keep the sparkle end unclipped
        if lo < hi:
            return mcolors.LogNorm(vmin=lo, vmax=hi)
    lo, hi = np.percentile(vals, ROBUST_PCT)
    if lo == hi:
        lo, hi = vals.min(), vals.max()
    return mcolors.Normalize(vmin=lo, vmax=hi)


def _panel(ax, xyz, norm, cmap):
    """One pcolormesh panel (plain axes, gray land, zero-clip).
    Inputs: ax; xyz ((x, y, arr) or None); norm; cmap.
    Outputs: the QuadMesh or None (placeholder).
    Generated by LH and Claude
    """
    ax.set_facecolor(LAND_COLOR)
    ax.set_xticks([])
    ax.set_yticks([])
    if xyz is None:
        ax.text(0.5, 0.5, "n/a", ha="center", va="center",
                fontsize=9, style="italic",
                transform=ax.transAxes)
        return None
    x, y, arr = xyz
    if isinstance(norm, mcolors.LogNorm):
        arr = np.clip(arr, norm.vmin, None)   # zeros -> bottom colour
    return ax.pcolormesh(x, y, arr, norm=norm, cmap=cmap,
                         shading="nearest")


def stage_grid(labels, top_xyz, bot_xyz, *, cmap, suptitle):
    """4-row grid: rows 1-2 = top variant (full/zoom), rows 3-4 =
    bottom variant; columns = labels; shared per-column norms.
    Inputs: labels (list[str]); top_xyz/bot_xyz (dict label ->
    (x, y, arr) or None); cmap; suptitle.
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    n = len(labels)
    fig, axes = plt.subplots(
        4, n, figsize=(PANEL_W * n, PANEL_H * 4 + 0.8),
        squeeze=False)
    for col, lab in enumerate(labels):
        t_full, b_full = top_xyz.get(lab), bot_xyz.get(lab)
        t_zoom = (regions.crop_zoom(*t_full, REGION,
                                    half_km=ZOOM_HALF_KM)
                  if t_full is not None else None)
        b_zoom = (regions.crop_zoom(*b_full, REGION,
                                    half_km=ZOOM_HALF_KM)
                  if b_full is not None else None)
        norm = _stage_norm(
            [t_full[2] if t_full is not None else None,
             b_full[2] if b_full is not None else None])
        mappable = None
        for row, xyz in enumerate((t_full, t_zoom, b_full,
                                   b_zoom)):
            pm = _panel(axes[row, col], xyz, norm, cmap)
            mappable = pm or mappable
        axes[0, col].set_title(lab, fontsize=11)
        if mappable is not None:
            fig.colorbar(mappable, ax=list(axes[:, col]),
                         orientation="horizontal", shrink=0.85,
                         pad=0.02, aspect=30)
    for row, label in enumerate(ROW_LABELS):
        axes[row, 0].set_ylabel(label, fontsize=10)
    fig.suptitle(suptitle, fontsize=13)
    plt.show()


def ecco_vs_sqfirst(field, cmap=cmo.amp):
    """Tracer-grad² A/B: raw, (d/dx)², (d/dy)², final.
    Component columns: ECCO rows = geographic, sq-first rows =
    MODEL-axis (see header caveat — swapped on rotated faces).
    Inputs: field (str, key into TRACERS); cmap.
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    raw = TRACERS[field][0]
    labels = [raw, "(d/dx)2", "(d/dy)2", field]
    ecco = {raw: live_region[raw],
            "(d/dx)2": live_region[f"{field}_dx2_ecco"],
            "(d/dy)2": live_region[f"{field}_dy2_ecco"],
            field: live_region[f"{field}_ecco"]}
    sqf = {raw: live_region[raw],
           "(d/dx)2": live_region[f"{field}_dx2_sqf"],
           "(d/dy)2": live_region[f"{field}_dy2_sqf"],
           field: live_region[f"{field}_sqfirst"]}
    stage_grid(labels, ecco, sqf, cmap=cmap,
               suptitle=(f"{field} \u2014 ECCO (rows 1-2) vs "
                         f"square-first (rows 3-4), {REGION}"))


def kin_vs_sqfirst(field, cmap=cmo.amp):
    """Kinematic A/B (single column): ECCO vs C-grid-native.
    Inputs: field ('strain_mag' or 'okubo_weiss'); cmap.
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    stage_grid(
        [field], {field: live_region[f"{field}_ecco"]},
        {field: live_region[f"{field}_sqfirst"]}, cmap=cmap,
        suptitle=(f"{field} \u2014 ECCO vs C-grid-native "
                  f"sq-first, {REGION}"))

print("ready: ecco_vs_sqfirst(...), kin_vs_sqfirst(...)")

## Section 5 — Figures

Read each figure left to right: the ECCO zooms (row 2) show white
specks; the sq-first zooms (row 4) show the same fronts without
them.  Any white surviving in BOTH is a genuine critical point.

### gradtheta2

In [ ]:
ecco_vs_sqfirst("gradtheta2")

### gradsalt2

In [ ]:
ecco_vs_sqfirst("gradsalt2")

### gradeta2

Control: Eta is smooth at grid scale,
so the two variants should look
near-identical.

In [ ]:
ecco_vs_sqfirst("gradeta2")

### gradrho2

In [ ]:
ecco_vs_sqfirst("gradrho2")

### gradb2

In [ ]:
ecco_vs_sqfirst("gradb2")

### strain_mag

In [ ]:
kin_vs_sqfirst("strain_mag")

### okubo_weiss

Signed field — linear scale; look for the salt-and-pepper extremes calming.

In [ ]:
kin_vs_sqfirst("okubo_weiss")

## Section 6 — ECCO vs square-first: regional RMSE

Bulk size of the discretization difference per field.  Plain RMSE
is dominated by the large values (fronts, where the two forms
agree); the log-space RMSE weights the small values where the
cancellation speckle lives — expect it to be much larger.

In [ ]:
# Section 6: RMSE between the ECCO and sq-first variants (REGION).
_PAIRS = ([(f, f"{f}_ecco", f"{f}_sqfirst") for f in TRACERS]
          + [("strain_mag", "strain_mag_ecco",
              "strain_mag_sqfirst"),
             ("okubo_weiss", "okubo_weiss_ecco",
              "okubo_weiss_sqfirst")])

print(f"{'field':22s} {'RMSE':>11s} {'RMSE/RMS':>9s} "
      f"{'log10-RMSE':>11s}   ({REGION})")
for _name, _ke, _ks in _PAIRS:
    _, _, a = live_region[_ke]
    _, _, b = live_region[_ks]
    m = np.isfinite(a) & np.isfinite(b)
    rmse = float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))
    rms = float(np.sqrt(np.mean(b[m] ** 2)))
    # log-space RMSE (positive-definite fields only).
    mp = m & (a > 0) & (b > 0)
    lrmse = (float(np.sqrt(np.mean(
        (np.log10(a[mp]) - np.log10(b[mp])) ** 2)))
        if mp.sum() else float("nan"))
    print(f"{_name:22s} {rmse:11.3e} {100*rmse/rms:8.2f}% "
          f"{lrmse:11.3f}")
print("\nlog10-RMSE in decades; driven by the speckle pixels "
      "(small values), where the two stencils disagree most.")

## Findings (2026-08-05, gulf_stream)

- **Confirmed**: white specks in every ECCO grad² zoom are absent
  in the square-first rows; frontal structure identical.
- **gradeta2** (control): no specks either way — Eta has no
  grid-scale extrema for the interp to cancel.
- **strain_mag / okubo_weiss**: both improved by the C-grid-native
  square-first invariants.
- **Component swap**: corr(ECCO dx², sq-first dy²) ≈ 0.9 vs ≈ 0.5
  same-name on this (rotated) face — rotation invariance of the
  SUM verified; individual geographic component-squares are
  impossible square-first (the rotation cross term is destroyed
  by squaring).
- Production switched to square-first
  (`calculate_grad_squared_tracer`, `kinematic_invariants`) —
  see docs/Fields.md and prompts/field_validation.md.